# NLP Text Preprocessing

This notebook covers the preprocessing topics listed in `list_of_topics` using small, easy-to-follow examples.



Topics included:

- Introduction

- Lowercasing

- Remove HTML tags

- Remove URLs

- Remove punctuation

- Chat word treatment

- Spelling correction

- Removing stop words

- Handling emojis

- Tokenization

- Stemming

- Lemmatization



Each section uses simple text samples so the effect of each preprocessing step is clear.

In [1]:
# Import libraries and prepare small sample texts used across the notebook.

import re

import string



import emoji

import nltk

import pandas as pd

from bs4 import BeautifulSoup

from nltk.corpus import stopwords

from nltk.stem import PorterStemmer, WordNetLemmatizer

from nltk.tokenize import sent_tokenize, word_tokenize

from spellchecker import SpellChecker



# Download the small NLTK resources needed for tokenization and lemmatization.

nltk.download("punkt", quiet=True)

nltk.download("punkt_tab", quiet=True)

nltk.download("stopwords", quiet=True)

nltk.download("wordnet", quiet=True)

nltk.download("omw-1.4", quiet=True)



# Create reusable text samples.

raw_text = "<p>HELLO!!! Visit https://example.com ASAP 😄</p>"

chat_text = "LOL BRB, I will ping U ASAP"

misspelled_text = "speling corection in nlp is usefull"

emoji_text = "I love NLP 😄 but debugging 😓 can be slow 🚀"

token_text = "NLP is fun. It helps computers understand language!"

word_forms = ["running", "studies", "better", "playing"]



reviews = pd.DataFrame(

    {

        "review": [

            "<b>HELLO</b> I LOVED this movie!!! Visit www.example.com 😄",

            "LOL this product is gr8, but delivery was sloooow...",

            "NLP preprocessing helps in text-cleaning tasks.",

        ]

    }

)



print("Sample raw text:")

print(raw_text)

print("\nSample review data:")

display(reviews)

Sample raw text:
<p>HELLO!!! Visit https://example.com ASAP 😄</p>

Sample review data:


,review
0,<b>HELLO</b> I LOVED this movie!!! Visit www.e...
1,"LOL this product is gr8, but delivery was sloo..."
2,NLP preprocessing helps in text-cleaning tasks.


## 1. Introduction

Text preprocessing is the step where raw text is cleaned and normalized before feature extraction or modeling.



Common reasons for preprocessing:

- Make similar words look the same

- Remove noisy content such as HTML, URLs, and punctuation

- Prepare clean tokens for vectorizers and models



The important point is that preprocessing is task-dependent. A step that helps one model may hurt another.

In [2]:
# Show a few noisy examples before preprocessing.

examples = pd.DataFrame(

    {

        "type": ["raw_text", "chat_text", "misspelled_text", "emoji_text"],

        "text": [raw_text, chat_text, misspelled_text, emoji_text],

    }

)



display(examples)

,type,text
0,raw_text,<p>HELLO!!! Visit https://example.com ASAP 😄</p>
1,chat_text,"LOL BRB, I will ping U ASAP"
2,misspelled_text,speling corection in nlp is usefull
3,emoji_text,I love NLP 😄 but debugging 😓 can be slow 🚀


## 2. Lowercasing

Lowercasing is a text normalization step. The goal is to treat words with the same meaning as the same token.



Important note:

- In cased models, `HELLO` and `hello` can be treated differently.

- In uncased models, they are usually treated as the same token.

- Classical pipelines such as Bag of Words and TF-IDF usually benefit from lowercasing.

- Tasks like NER or some sentiment settings may keep case information because it can carry meaning.

In [3]:
# Lowercase text in a simple dataframe column.

lowercase_df = reviews.copy()

lowercase_df["review_lower"] = lowercase_df["review"].str.lower()



# Show how cased and uncased vocabularies differ.

cased_tokens = ["HELLO", "hello"]

uncased_tokens = [token.lower() for token in cased_tokens]



display(lowercase_df[["review", "review_lower"]])

print("Cased tokens:", cased_tokens)

print("Uncased tokens:", uncased_tokens)

,review,review_lower
0,<b>HELLO</b> I LOVED this movie!!! Visit www.e...,<b>hello</b> i loved this movie!!! visit www.e...
1,"LOL this product is gr8, but delivery was sloo...","lol this product is gr8, but delivery was sloo..."
2,NLP preprocessing helps in text-cleaning tasks.,nlp preprocessing helps in text-cleaning tasks.


Cased tokens: ['HELLO', 'hello']
Uncased tokens: ['hello', 'hello']


## 3. Remove HTML Tags

HTML tags often appear in scraped reviews, web pages, and user-generated content.



The goal is to keep the visible text and remove the markup. You can do this with regex for simple cases or with an HTML parser for more robust handling.

In [4]:
# Remove HTML tags with a simple regex and compare with BeautifulSoup.

def remove_html_tags_regex(text):

    pattern = re.compile(r"<.*?>")

    return pattern.sub("", text)





def remove_html_tags_parser(text):

    return BeautifulSoup(text, "html.parser").get_text(separator=" ", strip=True)





html_sample = "<div><h1>Hello</h1><p>This is <b>NLP</b>.</p></div>"

print("Regex result:", remove_html_tags_regex(html_sample))

print("Parser result:", remove_html_tags_parser(html_sample))

Regex result: HelloThis is NLP.
Parser result: Hello This is NLP .


## 4. Remove URLs

URLs usually add noise when the goal is to learn from the sentence meaning rather than the specific link.



A regex-based replacement works well for common `http`, `https`, and `www` patterns.

In [5]:
# Remove URLs from text with a regex pattern.

def remove_url(text):

    pattern = re.compile(r"https?://\S+|www\.\S+")

    return pattern.sub("", text)





url_sample = "Read more at https://example.com or www.demo.com for details."

print("Original:", url_sample)

print("Cleaned :", remove_url(url_sample))

Original: Read more at https://example.com or www.demo.com for details.
Cleaned : Read more at  or  for details.


## 5. Remove Punctuation

Punctuation can be useful in some tasks, but for many classical NLP pipelines it is removed to simplify the vocabulary.



Examples:

- `Hello!!!` becomes `Hello`

- `How's` may become `Hows` if punctuation is removed directly

In [6]:
# Remove punctuation with a regex pattern.

punctuation_sample = "Hello!!! How's NLP going? Great, right :)"

cleaned_punctuation = re.sub(r"[^\w\s]", "", punctuation_sample)



print("Original:", punctuation_sample)

print("Cleaned :", cleaned_punctuation)

Original: Hello!!! How's NLP going? Great, right :)
Cleaned : Hello Hows NLP going Great right 


## 6. Chat Word Treatment

Chat words and abbreviations are common in social media, SMS, and informal reviews.



Examples:

- `LOL` -> `Laughing Out Loud`

- `BRB` -> `Be Right Back`

- `ASAP` -> `As Soon As Possible`



Replacing them can make text easier for downstream models to understand.

In [13]:
# Replace a few common chat words with their expanded forms.

chat_words = {

    "LOL": "Laughing Out Loud",

    "BRB": "Be Right Back",

    "ASAP": "As Soon As Possible",

    "U": "You",

    "GR8": "Great",

}





def chat_conversion(text):

    converted_words = []

    for word in text.split():

        prefix = ""

        suffix = ""

        cleaned_word = word



        while cleaned_word and cleaned_word[0] in string.punctuation:

            prefix += cleaned_word[0]

            cleaned_word = cleaned_word[1:]



        while cleaned_word and cleaned_word[-1] in string.punctuation:

            suffix = cleaned_word[-1] + suffix

            cleaned_word = cleaned_word[:-1]



        expanded = chat_words.get(cleaned_word.upper(), cleaned_word)

        converted_words.append(prefix + expanded + suffix)



    return " ".join(converted_words)





print("Original:", chat_text)

print("Expanded:", chat_conversion(chat_text))

Original: LOL BRB, I will ping U ASAP
Expanded: Laughing Out Loud Be Right Back, I will ping You As Soon As Possible


## 7. Spelling Correction

Misspelled words can increase vocabulary size and hurt model quality.



Common Python tools include:

- `autocorrect`

- `pyspellchecker`

- `jamspell`



Below, the notebook uses `pyspellchecker` because it is simple and works well for demonstration.

In [14]:
# Correct misspelled words with pyspellchecker.

spell = SpellChecker()

spell.word_frequency.load_words(["nlp"])





def correct_spelling(text):

    corrected_words = []

    for word in text.split():

        corrected_words.append(spell.correction(word) or word)

    return " ".join(corrected_words)





print("Original:", misspelled_text)

print("Corrected:", correct_spelling(misspelled_text))

Original: speling corection in nlp is usefull
Corrected: spelling correction in nlp is useful


## 8. Removing Stop Words

Stop words are very common words such as `is`, `the`, `and`, and `in`.



In many classical pipelines they are removed because they add little meaning. In some tasks, however, they can still matter, so this is also task-dependent.

In [9]:
# Remove stop words with the NLTK English stop word list.

stop_words = set(stopwords.words("english"))

stopword_sample = "this is a simple example showing how stop words are removed in nlp"

filtered_words = [word for word in stopword_sample.split() if word.lower() not in stop_words]



print("Original:", stopword_sample)

print("Filtered:", " ".join(filtered_words))

Original: this is a simple example showing how stop words are removed in nlp
Filtered: simple example showing stop words removed nlp


## 9. Handling Emojis

There are two common options for emojis:

- Remove them when they are not useful for the task

- Replace them with text descriptions when they carry sentiment or meaning



For example, `😄` can be converted to `grinning_face_with_smiling_eyes`.

In [10]:
# Remove emojis or replace them with readable text.

def remove_emojis(text):

    return emoji.replace_emoji(text, replace="")





def replace_emojis(text):

    return emoji.demojize(text, delimiters=(" ", " "))





print("Original:", emoji_text)

print("Without emojis:", remove_emojis(emoji_text))

print("Emoji as text:", replace_emojis(emoji_text))

Original: I love NLP 😄 but debugging 😓 can be slow 🚀
Without emojis: I love NLP  but debugging  can be slow 
Emoji as text: I love NLP  grinning_face_with_smiling_eyes  but debugging  downcast_face_with_sweat  can be slow  rocket 


## 10. Tokenization

Tokenization means splitting text into smaller units.



Common options:

- `split()` for very basic tokenization

- Regular expressions for better control

- NLTK for standard educational tokenizers

- spaCy for fast production-ready tokenization



Every method has tradeoffs, so the best choice depends on the task and the text.

In [11]:
# Compare a few tokenization approaches.

split_tokens = token_text.split()

regex_tokens = re.findall(r"\b\w+\b", token_text)

sentence_tokens = sent_tokenize(token_text)

word_tokens = word_tokenize(token_text)



print("Using split():", split_tokens)

print("Using regex :", regex_tokens)

print("Sentence tokens:", sentence_tokens)

print("Word tokens:", word_tokens)

Using split(): ['NLP', 'is', 'fun.', 'It', 'helps', 'computers', 'understand', 'language!']
Using regex : ['NLP', 'is', 'fun', 'It', 'helps', 'computers', 'understand', 'language']
Sentence tokens: ['NLP is fun.', 'It helps computers understand language!']
Word tokens: ['NLP', 'is', 'fun', '.', 'It', 'helps', 'computers', 'understand', 'language', '!']


In [16]:
# Use spaCy's lightweight English tokenizer.

import spacy



spacy_tokenizer = spacy.blank("en")

spacy_doc = spacy_tokenizer(token_text)

spacy_tokens = [token.text for token in spacy_doc]



print("spaCy tokens:", spacy_tokens)

spaCy tokens: ['NLP', 'is', 'fun', '.', 'It', 'helps', 'computers', 'understand', 'language', '!']


## 11. Stemming and 12. Lemmatization

Stemming and lemmatization are both normalization techniques that reduce words to a base form, but they work differently.



**Stemming**

- Fast and simple

- May return forms that are not real dictionary words

- Example: `studies` -> `studi`



**Lemmatization**

- Uses vocabulary and grammar knowledge

- Produces real base words more often

- Example: `studies` -> `study`



The next code cell shows both methods side by side.

In [15]:
# Compare stemming and lemmatization side by side.

stemmer = PorterStemmer()

lemmatizer = WordNetLemmatizer()

lemma_pos_map = {"running": "v", "studies": "v", "better": "a", "playing": "v"}



comparison_df = pd.DataFrame(

    {

        "word": word_forms,

        "stemmed": [stemmer.stem(word) for word in word_forms],

        "lemmatized": [lemmatizer.lemmatize(word, pos=lemma_pos_map[word]) for word in word_forms],

    }

)



display(comparison_df)

,word,stemmed,lemmatized
0,running,run,run
1,studies,studi,study
2,better,better,good
3,playing,play,play
